# NB-03: T-6 脚質分類（逃/先/差/追）

**ターゲット**: 各馬が「逃・先・差・追」のどの脚質で走るかを予測する。  
**モデル**: LightGBM 4クラス分類 (num_class=4)  
**主評価指標**: Accuracy, F1-macro  
**注意**: T-6 出力は T-1（勝率モデル）の入力特徴量になるため **Stage 1** で先行実行する。

### ラベル導出
`passing_order`（例: `"4-3-3-2"`）の平均通過順位を頭数で正規化して分類。
- 逃: avg_pos <= 1.5
- 先: avg_pos <= field_size × 0.30
- 差: avg_pos <= field_size × 0.65
- 追: それ以上


In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/work/keiba-vpn")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.pipeline.models.notebook_utils import (
    load_master, load_raw_results, encode_cats, feature_set,
    train_lgb, oof_predict, eval_classification, save_oof,
    DEFAULT_PARAMS_MULTICLASS, FEAT_RACE, FEAT_HORSE, FEAT_JK_BASE, FEAT_TR_BASE,
    derive_running_style, SURFACE_CATS,
)
print("imports OK")


## 1. データ読み込みとラベル生成

In [ ]:
# ── マスターデータ ────────────────────────────────────────────────────────
df = load_master()

# ── passing_order を race_result_flat から取得 ────────────────────────────
rr = load_raw_results(["race_id","horse_number","passing_order","field_size"])
rr = rr.drop_duplicates(subset=["race_id","horse_number"])

df = df.merge(rr[["race_id","horse_number","passing_order"]], on=["race_id","horse_number"], how="left")

# ── 脚質ラベル生成 ────────────────────────────────────────────────────────
df["running_style"] = [
    derive_running_style(po, fs)
    for po, fs in zip(df["passing_order"], df["field_size"])
]
label_map = {"逃": 0, "先": 1, "差": 2, "追": 3}
rev_map   = {v: k for k, v in label_map.items()}

print("脚質分布:")
print(df["running_style"].value_counts())
print(f"\nラベルなし率: {df['running_style'].isna().mean():.1%}")
df["surface_cat"] = df["surface_cat"].astype(str)  # Categorical → str


## 2. 特徴量定義・前処理

In [ ]:
# T-6 専用特徴量: 通過順関連を重視
FEAT_T6 = feature_set(df, extra=[
    "jk_prior_all_avg_pass_first", "jk_prior_all_avg_pass_norm_first",
    "jk_roll10_avg_pass_first", "jk_roll5_avg_pass_first",
    "tr_prior_all_avg_pass_first",
])
print(f"特徴量数: {len(FEAT_T6)}")
print(FEAT_T6[:15])

# categorical 列
CAT_USE = [c for c in ["venue","surface","direction","grade","track_condition","weather","sex"] if c in df.columns]
df = encode_cats(df, CAT_USE)


## 3. 馬場別モデル学習 + OOF 予測

In [ ]:
params = {**DEFAULT_PARAMS_MULTICLASS, "num_class": 4}
oof_all = pd.DataFrame()
models: dict = {}

for sc in SURFACE_CATS:
    df_sc = df[df["surface_cat"] == sc].copy()
    df_sc = df_sc[df_sc["running_style"].notna()]
    df_tr = df_sc[df_sc["split"] == "train"]
    df_vl = df_sc[df_sc["split"] == "valid"]

    print(f"\n=== {sc} ===  train={len(df_tr):,}  valid={len(df_vl):,}")
    if len(df_tr) < 100:
        print("  スキップ（データ不足）")
        continue

    # 学習
    model = train_lgb(df_tr, df_vl, FEAT_T6, "running_style", params,
                      cat_features=CAT_USE, label_encoder=label_map)
    models[sc] = model

    # valid 評価
    X_vl = df_vl[FEAT_T6]
    pred_proba = model.predict(X_vl)             # shape (n, 4)
    pred_class = pred_proba.argmax(axis=1)
    y_vl = df_vl["running_style"].map(label_map)
    mask = y_vl.notna()
    metrics = eval_classification(y_vl[mask], pred_class[mask.values], task="multiclass")
    print(f"  Valid metrics: {metrics}")

    # OOF
    oof = oof_predict(df_tr, FEAT_T6, "running_style", params,
                      cat_features=CAT_USE, label_encoder=label_map)
    oof_df = df_tr[["race_id","horse_number","surface_cat","running_style"]].copy()
    oof_df["t6_oof_class"] = oof.values
    # 各クラス確率も保存するため全データ予測
    proba_tr = model.predict(df_tr[FEAT_T6])
    for ci, label in enumerate(["逃","先","差","追"]):
        oof_df[f"t6_prob_{label}"] = proba_tr[:, ci]
    oof_all = pd.concat([oof_all, oof_df], ignore_index=True)

print("\nモデル学習完了:", list(models.keys()))


## 4. テストセット評価 & 混同行列

In [ ]:
for sc, model in models.items():
    df_te = df[(df["surface_cat"] == sc) & (df["split"] == "test") & df["running_style"].notna()]
    if df_te.empty:
        continue
    pred_class = model.predict(df_te[FEAT_T6]).argmax(axis=1)
    y_te = df_te["running_style"].map(label_map).values
    metrics = eval_classification(y_te, pred_class, task="multiclass")
    print(f"[{sc}] Test: {metrics}")

    cm = confusion_matrix(y_te, pred_class, labels=[0,1,2,3])
    labels_str = ["逃","先","差","追"]
    disp = ConfusionMatrixDisplay(cm, display_labels=labels_str)
    fig, ax = plt.subplots(figsize=(5,4))
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f"T-6 Confusion Matrix [{sc}]")
    plt.tight_layout()
    plt.savefig(f"/home/jovyan/work/keiba-vpn/data/local/modeling/oof/t6_cm_{sc}.png", dpi=80)
    plt.show()


## 5. OOF 保存 & 特徴量重要度

In [ ]:
if not oof_all.empty:
    save_oof(oof_all, "t6_prob_逃", "t6", key_cols=["race_id","horse_number","surface_cat",
                                                     "t6_oof_class","t6_prob_逃","t6_prob_先","t6_prob_差","t6_prob_追"])
    # 全列を含む形で別途保存
    oof_all.to_parquet("/home/jovyan/work/keiba-vpn/data/local/modeling/oof/t6_oof.parquet", index=False)
    print(f"OOF 保存: shape={oof_all.shape}")

# 特徴量重要度（最大モデル）
best_sc = max(models, key=lambda s: models[s].num_trees()) if models else None
if best_sc:
    imp = pd.Series(models[best_sc].feature_importance(importance_type="gain"),
                    index=FEAT_T6).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(8,6))
    imp.head(20).plot.barh(ax=ax)
    ax.set_title(f"T-6 Feature Importance [{best_sc}]")
    plt.tight_layout()
    plt.savefig("/home/jovyan/work/keiba-vpn/data/local/modeling/oof/t6_feature_importance.png", dpi=80)
    plt.show()
    print("Top 10 features:")
    print(imp.head(10))
